In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random
from PIL import Image
import pandas as pd
import ast
import matplotlib.gridspec as gridspec

from DWT import DWT_2D
from IXCM import IXCNet

import os

In [2]:
def visualize_evidence_explainability(models_dict, dwt_module, val_loader, class_names, device, sample_idx=0, batch_id=1, save_dir=''):
    for name, model in models_dict.items():
        model.eval()
    dwt_module.eval()

    target_batch_idx = batch_id 
    images = None
    labels = None
    
    for i, (imgs, lbls) in enumerate(val_loader):
        if i == target_batch_idx:
            images = imgs
            labels = lbls
            break 
    
    if images is None:
        print("Error: Batch ID out of range.")
        return

    images = images.to(device)
    b_idx = sample_idx
    
    with torch.no_grad():
        s_tensor, lh, hl, hh = dwt_module(images)
        a_tensor = lh + hl

        alpha_s_list = []
        beta_s_list = []
        alpha_a_list = []
        beta_a_list = []
        
        for cls_name in class_names:
            if cls_name not in models_dict:
                print(f"Warning: Model for {cls_name} not found in models_dict.")
                alpha_s_list.append(1.0); beta_s_list.append(1.0)
                alpha_a_list.append(1.0); beta_a_list.append(1.0)
                continue

            curr_model = models_dict[cls_name]
            output = curr_model(images)

            alphas = output[0][b_idx].cpu().numpy() # Shape: [2]
            betas  = output[1][b_idx].cpu().numpy() # Shape: [2]
            
            alpha_s_list.append(alphas[0])
            alpha_a_list.append(alphas[1])
            beta_s_list.append(betas[0])
            beta_a_list.append(betas[1])

    alpha_s_vals = np.array(alpha_s_list)
    beta_s_vals  = np.array(beta_s_list)
    alpha_a_vals = np.array(alpha_a_list)
    beta_a_vals  = np.array(beta_a_list)

    prob_s = alpha_s_vals / (alpha_s_vals + beta_s_vals + 1e-6)
    prob_a = alpha_a_vals / (alpha_a_vals + beta_a_vals + 1e-6)

    def to_img(t):
        tensor_img = t[b_idx, 0, :, :]  
        img = tensor_img.cpu().numpy()
        img_min = img.min()
        img_max = img.max()
        if img_max - img_min > 1e-6:
            img = (img - img_min) / (img_max - img_min)
        return img

    img_orig = to_img(images)
    img_s = to_img(s_tensor)
    img_a = to_img(a_tensor)
    
    curr_labels = labels[b_idx].numpy()
    num_classes = len(class_names)
    
    fig = plt.figure(figsize=(18, 12))

    outer_gs = gridspec.GridSpec(2, 1, height_ratios=[1.5, 0.8], hspace=0.01)

    top_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=outer_gs[0], wspace=0.2)
    
    ax_orig = fig.add_subplot(top_gs[0])
    ax_s = fig.add_subplot(top_gs[1])
    ax_a = fig.add_subplot(top_gs[2])

    ax_orig.imshow(img_orig, cmap='gray')
    gt_classes = [class_names[i] for i in range(num_classes) if curr_labels[i]==1]
    gt_str = ",".join(gt_classes) if gt_classes else "None"
    ax_orig.set_title(f"Original CXR Image\nDisease(s): \n[{gt_str}]", fontsize=16, fontweight='bold')
    ax_orig.axis('off')
    
    ax_s.imshow(img_s, cmap='gray')
    ax_s.set_title("Structure (S)", fontsize=30, fontweight='bold', color='#1f77b4')
    ax_s.axis('off')
    
    ax_a.imshow(img_a, cmap='gray')
    ax_a.set_title("Texture (T)", fontsize=30, fontweight='bold', color='#d62728')
    ax_a.axis('off')

    bottom_gs = gridspec.GridSpecFromSubplotSpec(1, 3, 
                                                 subplot_spec=outer_gs[1], 
                                                 width_ratios=[1, 0.25, 1]) 
    
    ax_bar_s = fig.add_subplot(bottom_gs[0])
    ax_bar_a = fig.add_subplot(bottom_gs[2])

    def plot_bar(ax, alpha_vals, beta_vals, probs, color, title):
        y_pos = np.arange(num_classes)
        bars = ax.barh(y_pos, probs, align='center', color=color, alpha=0.85, edgecolor='white')
        
        ax.set_yticks(y_pos)

        ax.set_yticklabels(class_names, fontsize=20, fontweight='medium')
        ax.invert_yaxis()
        if title.split(' ')[0] == 'Structure':
            ax.set_xlabel('Structural Positive Probability', fontsize=20)
        else:
            ax.set_xlabel('Textural Positive Probability', fontsize=20)
        ax.set_xlim(0, 1.0)
        ax.tick_params(axis='x', labelsize=15)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        ax.set_title(title, fontsize=26, fontweight='bold', pad=15)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        
        for i, bar in enumerate(bars):
            is_gt = curr_labels[i] == 1
            font_w = 'bold' if is_gt else 'normal'
            txt_color = 'black' if is_gt else '#555555'
            label_text = f" {probs[i]:.2f} (α={alpha_vals[i]:.2f},β={beta_vals[i]:.2f})"
            bar_width = bar.get_width()
            if bar_width > 0.6:
                ax.text(bar_width - 0.02, bar.get_y() + bar.get_height()/2, 
                        label_text, 
                        va='center', ha='right', 
                        fontsize=18, fontweight=font_w, color=txt_color)
            else:
                ax.text(bar_width + 0.01, bar.get_y() + bar.get_height()/2, 
                        label_text, 
                        va='center', ha='left', 
                        fontsize=18, fontweight=font_w, color=txt_color)

    plot_bar(ax_bar_s, alpha_s_vals, beta_s_vals, prob_s, '#89CFF0', "Structural Evidence")

    plot_bar(ax_bar_a, alpha_a_vals, beta_a_vals, prob_a, '#F4C2C2', "Textural Evidence")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, f'{sample_idx}_{batch_id}.png'), dpi=300)
    plt.show()

In [3]:
DEVICE = 'cuda'

LABEL_COLUMNS = [
    'Cardiomegaly', 'Pleural Effusion', 'Pneumonia'
]

class MIMICXrayDataset(Dataset):

    def __init__(self, dataframe, transform=None, class_name=['Atelectasis']):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(self.dataframe[class_name].values, dtype=torch.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_paths_list = self.dataframe.iloc[idx]['image_paths_list']
        labels = self.labels[idx]

        selected_path = None
        pa_paths = [p for p in image_paths_list if 'PA' in p]
        
        if len(pa_paths) > 0:
            selected_path = pa_paths[0]
        else:
            ap_paths = [p for p in image_paths_list if 'AP' in p]
            if len(ap_paths) > 0:
                selected_path = ap_paths[0]
            else:
                selected_path = image_paths_list[0]

        image = Image.open(selected_path).convert('L')

        if self.transform:
            image = self.transform(image)

        return image, labels

In [ ]:
print(f"{'='*20} Step 1: Initialize with Official Validation Set {'='*20}")

split_csv_path = ''
split_df = pd.read_csv(split_csv_path)

common_val_df = split_df[split_df['split'] == 'validate'][['study_id']]

models_dict = {}

print(f"\n{'='*20} Step 2: Cascading Intersection (Inner Merge) {'='*20}")

for idx, class_name in enumerate(LABEL_COLUMNS):
    csv_name = f"dataset_{class_name.replace(' ', '_')}.csv"
    csv_path = os.path.join('', csv_name)
    
    if not os.path.exists(csv_path):
        print(f"  [Error] File not found: {csv_path}. Intersection will fail for this class!")
        continue

    current_df = pd.read_csv(csv_path)

    if 'image_paths_list' not in common_val_df.columns:
        cols_to_use = ['study_id', class_name, 'image_paths_list']
    else:
        cols_to_use = ['study_id', class_name]

    current_df = current_df[[c for c in cols_to_use if c in current_df.columns]]

    common_val_df = pd.merge(common_val_df, current_df, on='study_id', how='inner')
    
    print(f"  - ({idx+1}/{len(LABEL_COLUMNS)}) Merged {class_name}: Remaining samples = {len(common_val_df)}")

    model = IXCNet(evi_vec_dim=64, wavename='haar', ydim=1, in_channels=1, img_dec=True).to(DEVICE)

    ckpt_name = "best_model_auc.pth"
    ckpt_path = os.path.join(f'', ckpt_name)
    
    if os.path.exists(ckpt_path):
        try:
            checkpoint = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(checkpoint)
            model.to(DEVICE)
            model.eval()
            models_dict[class_name] = model
            print(f"  -> Success: Model loaded from {ckpt_name}")
            
        except Exception as e:
            print(f"  -> Error loading {class_name}: {e}")
    else:
        print(f"  -> Warning: Checkpoint not found at {ckpt_path}")

print(f"\n{'='*20} Step 3: Post-processing {'='*20}")

if 'image_paths_list' not in common_val_df.columns:
    print("[Error] No image paths found! Please check if the first CSV file contains 'image_paths_list'.")
else:
    common_val_df = common_val_df.dropna(subset=['image_paths_list'])

def parse_and_fix_paths(path_str):
    if pd.isna(path_str): return []
    if isinstance(path_str, list):
        path_list = path_str
    else:
        try:
            path_list = ast.literal_eval(path_str)
        except:
            return []
            
    OLD_ROOT = ''
    NEW_ROOT = ''
    return [p.replace(OLD_ROOT, NEW_ROOT) for p in path_list]

if 'image_paths_list' in common_val_df.columns:
    common_val_df['image_paths_list'] = common_val_df['image_paths_list'].apply(parse_and_fix_paths)

missing_labels = [col for col in LABEL_COLUMNS if col not in common_val_df.columns]
if missing_labels:
    print(f"  [Warning] Missing labels in final dataset: {missing_labels}")
else:
    print(f"  [Success] All labels present.")


In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize(540),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

common_val_dataset = MIMICXrayDataset(
    dataframe=common_val_df,
    transform=val_transforms,
    class_name=LABEL_COLUMNS
)

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed + worker_id)
    random.seed(worker_seed + worker_id)

common_val_loader = DataLoader(
    common_val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    worker_init_fn=worker_init_fn
)

print("DataLoader Ready.")

In [ ]:
save_dir = ''

In [ ]:
wavename = 'haar'
dwt2D = DWT_2D(wavename).to(DEVICE)
visualize_evidence_explainability(models_dict, dwt2D, common_val_loader, LABEL_COLUMNS, DEVICE, sample_idx=0, batch_id=1, save_dir=save_dir)